<a href="https://colab.research.google.com/github/watanabe3tipapa/GoogleColaboratory/blob/main/mcp/colab_mcp_cms_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab MCP Server を Web サーバーとして公開し、PWA クライアントから CMS を構成する手法Google Colab 上の MCP Server を Streamable HTTP 対応の Web サービスとしてトンネリング公開し、Progressive Web App（PWA）クライアントからコンテンツ管理システム（CMS）を構築する完全ガイド。最終更新: 2026年9月8日

## 全体アーテクチャ本手法は、Google Colab の一時的な実行環境を「ヘッドレス CMS バックエンド」として位置づけ、PWA クライアントから Model Context Protocol（MCP）を介してコンテンツを管理する構成です。```┌─────────────────┐     ┌─────────────────────────────┐     ┌─────────────────┐│   PWA (Client)  │◄───►│      Public URL (ngrok)     │◄───►│  Google Colab   ││   - React/Vue   │ HTTP │   - Streamable HTTP / SSE   │Tunnel│  - FastAPI     ││ - ServiceWorker │ MCP  │      - CORS enabled         │      │  - MCP Server   ││  - IndexedDB    │      │   - Bearer/OAuth auth       │      │ - Google Drive  │└─────────────────┘     └─────────────────────────────┘     │  (persistence)  │                                                            └─────────────────┘```**ポイント**- Colab のセッションは一時的ですが、記事データを Google Drive に JSON 形式で永続化することで、セッション切断後もデータを保持できます。- MCP の Streamable HTTP トランスポートにより、PWA は標準的な HTTP 通信でサーバーとやり取りできます。

## 1. 必要なライブラリのインストールColab ノートブックのセルで以下を実行し、FastAPI と FastMCP、トンネリング用の pyngrok をインストールします。

In [ ]:
# Colab セルで実行!pip install fastapi uvicorn fastmcp pyngrok

## 2. バックエンド: FastAPI + FastMCP で Streamable HTTP Server を構築`fastmcp` と `FastAPI` を組み合わせ、MCP Server を Web アプリとしてマウントします。以下は、記事の CRUD とメディア管理を備えた CMS ツールを定義した完全な実装例です。

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Headerfrom fastapi.middleware.cors import CORSMiddlewarefrom fastmcp import FastMCPfrom fastmcp.integrations.fastapi import mount_mcpimport uvicornimport jsonimport osfrom datetime import datetimefrom google.colab import drive# Google Drive をマウント（永続化用）drive.mount('/content/drive')CMS_DIR = "/content/drive/MyDrive/cms_data"os.makedirs(CMS_DIR, exist_ok=True)app = FastAPI(title="Colab CMS MCP Server")# CORS: PWA からのアクセスを許可app.add_middleware(    CORSMiddleware,    allow_origins=["*"],  # 本番では PWA のドメインのみに制限    allow_credentials=True,    allow_methods=["*"],    allow_headers=["*"],)# MCP Server 定義mcp = FastMCP("ColabCMS")# ========================# CMS Tools (MCP)# ========================@mcp.tool()def list_posts() -> str:    """記事一覧を取得"""    posts = []    for fname in os.listdir(CMS_DIR):        if fname.endswith(".json"):            with open(os.path.join(CMS_DIR, fname), "r") as f:                posts.append(json.load(f))    return json.dumps(sorted(posts, key=lambda x: x.get("updated", ""), reverse=True))@mcp.tool()def get_post(slug: str) -> str:    """指定スラッグの記事を取得"""    path = os.path.join(CMS_DIR, f"{slug}.json")    if not os.path.exists(path):        return json.dumps({"error": "not_found"})    with open(path, "r") as f:        return json.dumps(json.load(f))@mcp.tool()def create_post(slug: str, title: str, content: str, tags: list = None) -> str:    """新規記事を作成"""    path = os.path.join(CMS_DIR, f"{slug}.json")    if os.path.exists(path):        return json.dumps({"error": "already_exists"})    data = {        "slug": slug,        "title": title,        "content": content,        "tags": tags or [],        "created": datetime.now().isoformat(),        "updated": datetime.now().isoformat(),    }    with open(path, "w") as f:        json.dump(data, f, ensure_ascii=False)    return json.dumps({"status": "created", "slug": slug})@mcp.tool()def update_post(slug: str, title: str = None, content: str = None, tags: list = None) -> str:    """記事を更新"""    path = os.path.join(CMS_DIR, f"{slug}.json")    if not os.path.exists(path):        return json.dumps({"error": "not_found"})    with open(path, "r") as f:        data = json.load(f)    if title:   data["title"] = title    if content: data["content"] = content    if tags:    data["tags"] = tags    data["updated"] = datetime.now().isoformat()    with open(path, "w") as f:        json.dump(data, f, ensure_ascii=False)    return json.dumps({"status": "updated", "slug": slug})@mcp.tool()def delete_post(slug: str) -> str:    """記事を削除"""    path = os.path.join(CMS_DIR, f"{slug}.json")    if os.path.exists(path):        os.remove(path)        return json.dumps({"status": "deleted", "slug": slug})    return json.dumps({"error": "not_found"})@mcp.tool()def list_media() -> str:    """アップロード済みメディア一覧"""    media_dir = os.path.join(CMS_DIR, "media")    os.makedirs(media_dir, exist_ok=True)    return json.dumps(os.listdir(media_dir))# FastAPI に MCP をマウント (Streamable HTTP)mount_mcp(app, mcp, path="/mcp")@app.get("/")def root():    return {"message": "Colab CMS MCP Server is running"}

### 実装のポイント| 項目 | 内容 ||------|------|| 永続化 | Colab のセッションは一時的なので、記事データは Google Drive に JSON 形式で保存 || トランスポート | `mount_mcp` で Streamable HTTP（POST/GET + SSE）を有効化。PWA から直接 MCP プロトコルで通信可能 || 公開 | `pyngrok` で `localhost:8000` をインターネット公開。PWA はこの HTTPS URL に接続 || 認証 | 必要に応じて `Authorization` ヘッダーを検証する Depends を `/mcp` やツール内に追加 |

## 3. サーバーの起動FastAPI サーバーをバックグラウンドで起動します。

In [ ]:
# FastAPI サーバーをバックグラウンドで起動!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &!sleep 2!cat server.log

## 4. 公開: ngrok でインターネットに出す2023年以降、ngrok の無料アカウントにも 1つの static dev domain（例: `xxx.ngrok-free.dev`）が割り当てられます。セッションを再起動しても URL が変わらないため、PWA 側のエンドポイントを固定できます。**ngrok 無料版の制約**- データ転送は約 1GB/月- リクエスト数は約 2万/月- HTTP/S エンドポイントにインタースティシャル広告ページが入ることがあります

In [ ]:
from pyngrok import ngrok# 自分の static domain を指定（ダッシュボードで確認可能）public_url = ngrok.connect(    addr="8000",    proto="http",    domain="your-domain.ngrok-free.dev"  # 無料で割り当てられたドメイン)print("Public URL:", public_url)  # https://your-domain.ngrok-free.app/mcp で常にアクセス可能

## 5. 代替: Cloudflare Tunnel（cloudflared）での公開ngrok よりも安定しており、挿入広告ページがありません。Colab 上で `cloudflared` を使う方法が広く使われています。

In [ ]:
# cloudflared のダウンロードとインストール!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared!chmod +x cloudflared# Cloudflare Tunnel で公開（ランダム一時URL）# 注意: 以下はフォアグラウンド実行なので、別セルで実行するか nohup を使ってください# !./cloudflared tunnel --url http://localhost:8000# バックグラウンド実行の例!nohup ./cloudflared tunnel --url http://localhost:8000 > cloudflared.log 2>&1 &!sleep 5!grep -o 'https://[a-zA-Z0-9-]\+\.trycloudflare\.com' cloudflared.log | head -1

### 公開手法の比較| 比較項目 | ngrok | Cloudflare Tunnel ||----------|-------|-------------------|| 無料固定ドメイン | 1つ（`*.ngrok-free.dev`） | カスタムドメイン紐付け可能 || 挿入広告ページ | あり（無料版） | なし || セットアップ難易度 | 簡単（pyngrok 1行） | やや手順が多い || 速度・安定性 | 良好 | 非常に良好 || Colab 相性 | 良好 | 良好 |

## 6. セキュリティ: Bearer Token 認証公開 URL は全世界に晒されるため、`/mcp` エンドポイントに Bearer Token 認証を適用します。

In [ ]:
from fastapi import Depends, HTTPException, Headerimport osAPI_TOKEN = os.environ.get("MCP_AUTH_TOKEN", "change-me-in-production")def verify_token(authorization: str = Header(None)):    if not authorization or not authorization.startswith("Bearer "):        raise HTTPException(status_code=401, detail="Missing or invalid token")    token = authorization.replace("Bearer ", "")    if token != API_TOKEN:        raise HTTPException(status_code=403, detail="Invalid token")    return token# 使用例: mount_mcp の前にエンドポイント保護を設定する場合# from fastapi import APIRouter# router = APIRouter()# @router.post("/mcp", dependencies=[Depends(verify_token)])# async def protected_mcp():#     passprint("API_TOKEN:", API_TOKEN)

## 7. フロントエンド: PWA からの接続### 7.1 SDK のインストールPWA 側は公式の TypeScript SDK（`@modelcontextprotocol/sdk`）を使い、`StreamableHTTPClientTransport` で接続します。```bashnpm install @modelcontextprotocol/sdk```### 7.2 MCP クライアントの実装```typescript// mcp-client.tsimport { Client, StreamableHTTPClientTransport } from "@modelcontextprotocol/sdk";const MCP_URL = "https://xxxx.ngrok-free.app/mcp"; // Colab で発行された URLlet client: Client | null = null;export async function connectMCP() {  const transport = new StreamableHTTPClientTransport(new URL(MCP_URL));  client = new Client({ name: "pwa-cms-client", version: "1.0.0" });  await client.connect(transport);  // ツール一覧を取得（CMS 機能の発見）  const tools = await client.listTools();  console.log("Available tools:", tools);  return client;}// CMS 操作ラッパーexport async function listPosts() {  if (!client) throw new Error("Not connected");  const result = await client.callTool("list_posts", {});  return JSON.parse(result.content[0].text);}export async function getPost(slug: string) {  if (!client) throw new Error("Not connected");  const result = await client.callTool("get_post", { slug });  return JSON.parse(result.content[0].text);}export async function createPost(  slug: string, title: string, content: string, tags?: string[]) {  if (!client) throw new Error("Not connected");  const result = await client.callTool("create_post", { slug, title, content, tags });  return JSON.parse(result.content[0].text);}export async function updatePost(  slug: string,  updates: { title?: string; content?: string; tags?: string[] }) {  if (!client) throw new Error("Not connected");  const result = await client.callTool("update_post", { slug, ...updates });  return JSON.parse(result.content[0].text);}export async function deletePost(slug: string) {  if (!client) throw new Error("Not connected");  const result = await client.callTool("delete_post", { slug });  return JSON.parse(result.content[0].text);}```### 7.3 PWA の構成例（React）```tsx// App.tsximport { useEffect, useState } from "react";import { connectMCP, listPosts, createPost } from "./mcp-client";export default function App() {  const [posts, setPosts] = useState<any[]>([]);  const [connected, setConnected] = useState(false);  useEffect(() => {    connectMCP().then(() => {      setConnected(true);      refresh();    });  }, []);  async function refresh() {    const data = await listPosts();    setPosts(data);  }  return (    <div className="p-4">      <h1>Colab CMS PWA</h1>      <p>Status: {connected ? "Connected" : "Connecting..."}</p>      <button onClick={refresh}>Refresh</button>      <ul>        {posts.map((p) => (          <li key={p.slug}>{p.title}</li>        ))}      </ul>    </div>  );}```### 7.4 PWA 化のポイント- **manifest.json**: アプリ名、アイコン、テーマカラーを設定- **Service Worker**: 記事のキャッシュ、オフライン閲覧を実現（IndexedDB や Cache API を併用）- **HTTPS 必須**: ngrok / Cloudflare Tunnel は自動的に HTTPS を提供するため、PWA 要件を満たします

## 8. まとめ本手法により、Google Colab の一時的な実行環境をヘッドレス CMS バックエンドとして活用し、PWA クライアントから MCP プロトコルでコンテンツ管理が可能になります。**主な利点:**- インフラコストゼロで CMS バックエンドを運用可能- Google Drive によるデータ永続化- Streamable HTTP トランスポートで標準的な Web 通信を利用- PWA によるモバイル対応とオフラインキャッシュ**注意事項:**- Colab セッションは一定時間で切断されるた、長期運用には定期的な再接続が必要- 公開 URL は適切に認証保護すること- 本番環境ではカスタムドメインと強固な認証を検討すること